# Empirical Investigation of Context-Window Degradation and Mitigation Strategies in Long-Document Synthesis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PCL2026001/awesome-Context-Window-Degradation-and-Its-Impact-on-Long-Document-Research-Synthesis/blob/main/PCL_Context_Window_Degradation_Experiment.ipynb)

**Course**: Post-Mid-Semester Assignment — AI Tools for Research (PCL Course)  
**Author**: Mayank Waila  
**Institution**: Indian Institute of Information Technology, Allahabad (IIITA)  
**Repository**: [GitHub Repository](https://github.com/PCL2026001/awesome-Context-Window-Degradation-and-Its-Impact-on-Long-Document-Research-Synthesis)  

---

## Abstract & Overview
Large Language Models (LLMs) frequently suffer from the *'Lost in the Middle'* phenomenon (Liu et al., 2024), where retrieval recall degrades significantly for evidence situated in the middle of long contexts compared to the boundaries (primacy and recency bias).

This reproducible Colab notebook implements and evaluates:
1. **Full-Context Baseline (FC-B1)**: Direct single-pass traversal under attention decay.
2. **Chunked Sequential Retrieval (CS-B2)**: Disjoint fixed sliding windows.
3. **Overlapping Chunks Retrieval (OC-B3)**: Stridden overlapping windows to avoid chunk-boundary truncation.
4. **Hierarchical Two-Pass Retrieval (H2P - Proposed)**: Coarse document summarization followed by localized high-resolution retrieval.

This notebook runs 100% deterministically, records real simulation trials ($N=200$ per cell, total 4,000 retrievals), generates tabular comparisons, and plots publication-grade figures.

### 1. Setup & Dependencies
The core simulation runs on Python standard libraries (`random`, `math`, `json`, `csv`). We also import `matplotlib` and `pandas` for publication-quality rendering and analysis in Google Colab.

In [ ]:
import random
import json
import csv
import math
import os
from typing import List, Tuple, Dict, Optional

import pandas as pd
import matplotlib.pyplot as plt

# Set plot styles for research publication quality
plt.rcParams.update({
    'font.size': 12,
    'figure.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print("Environment initialized successfully.")

### 2. Experimental Parameters & Attention Decay Formulation
We model the theoretical attention degradation observed in transformers using the U-shaped primacy-recency decay function:

$$w(i, N) = \max\left(0.05, 1.0 - \lambda \cdot \sin^2\left(\frac{\pi \cdot i}{N - 1}\right)\right)$$

where $\lambda$ denotes the decay intensity and $i / (N-1)$ is the normalized position in the document.

In [ ]:
# Experimental Configuration
RANDOM_SEED = 42
TOTAL_CHUNKS = 20
CHUNK_SIZE = 60          # Words per chunk
WINDOW_SIZE = 5          # Sliding window size in chunks
POSITIONS = [0, 5, 10, 15, 19]  # Normalized positions: Start, Early, Middle, Late, End
POSITION_LABELS = ["Start (0%)", "Early (25%)", "Middle (50%)", "Late (75%)", "End (100%)"]
NUM_TRIALS = 200         # 200 trials per cell for statistical significance
DISTRACTOR_POOL = 800    # Distractor vocabulary space

def attn_weight(pos: int, total: int, strength: float = 1.8) -> float:
    """
    U-shaped primacy-recency weight curve matching Liu et al. (2024).
    """
    if total <= 1:
        return 1.0
    frac = pos / (total - 1)
    w = 1.0 - strength * (math.sin(math.pi * frac) ** 2)
    return max(0.05, w)  # Clamped floor to represent residual signal

### 3. Data Synthesis & Retrieval Scorers

In [ ]:
def random_words(n: int, vocab_size: int = DISTRACTOR_POOL, rng: Optional[random.Random] = None) -> str:
    r = rng or random
    return " ".join(str(r.randint(0, vocab_size)) for _ in range(n))

def build_query_evidence(rng: random.Random) -> Tuple[str, str]:
    signal_tokens = ["SIGNAL_X", "SIGNAL_Y", "SIGNAL_Z"]
    partial_match = signal_tokens[:2]
    evidence = " ".join(signal_tokens) + " " + random_words(CHUNK_SIZE - len(signal_tokens), rng=rng)
    query = " ".join(partial_match)
    return query, evidence

def build_document(evidence: str, pos: int, rng: random.Random) -> List[str]:
    return [evidence if i == pos else random_words(CHUNK_SIZE, rng=rng) for i in range(TOTAL_CHUNKS)]

def bow_score(chunk: str, query: str) -> float:
    qt = set(query.lower().split())
    ct = set(chunk.lower().split())
    if not qt:
        return 0.0
    return len(qt & ct) / len(qt)

### 4. Implementation of Retrieval Methods
- **FC-B1**: Full-Context single pass under attention decay penalty.
- **CS-B2**: Chunked sequential window evaluation.
- **OC-B3**: Overlapping chunks window evaluation (stride = 2).
- **H2P**: Hierarchical two-pass (Coarse filter followed by Fine retrieval).

In [ ]:
def full_context(chunks: List[str], query: str, ev_pos: int, rng: random.Random) -> Tuple[float, float]:
    n = len(chunks)
    scores = [
        bow_score(c, query) * attn_weight(i, n) + rng.gauss(0, 0.015)
        for i, c in enumerate(chunks)
    ]
    pred = max(range(n), key=lambda i: scores[i])
    hit = int(pred == ev_pos)
    f1 = bow_score(chunks[pred], query) * hit
    return float(hit), f1

def chunked_sequential(chunks: List[str], query: str, ev_pos: int, rng: random.Random, window: int = WINDOW_SIZE) -> Tuple[float, float]:
    n = len(chunks)
    best_score, best_idx = -1.0, 0
    for start in range(0, n, window):
        win = chunks[start:start + window]
        w_len = len(win)
        for j, c in enumerate(win):
            s = bow_score(c, query) * attn_weight(j, w_len, strength=0.8) + rng.gauss(0, 0.015)
            if s > best_score:
                best_score, best_idx = s, start + j
    hit = int(best_idx == ev_pos)
    f1 = bow_score(chunks[best_idx], query) * hit
    return float(hit), f1

def overlapping_chunked(chunks: List[str], query: str, ev_pos: int, rng: random.Random, window: int = WINDOW_SIZE, stride: int = 2) -> Tuple[float, float]:
    n = len(chunks)
    best_score, best_idx = -1.0, 0
    starts = list(range(0, n - window + 1, stride)) + ([n - window] if n > window else [])
    for start in starts:
        win = chunks[start:start + window]
        w_len = len(win)
        for j, c in enumerate(win):
            s = bow_score(c, query) * attn_weight(j, w_len, strength=0.8) + rng.gauss(0, 0.015)
            if s > best_score:
                best_score, best_idx = s, start + j
    hit = int(best_idx == ev_pos)
    f1 = bow_score(chunks[best_idx], query) * hit
    return float(hit), f1

def hierarchical(chunks: List[str], query: str, ev_pos: int, rng: random.Random, coarse_win: int = WINDOW_SIZE, fine_k: int = 2) -> Tuple[float, float]:
    n = len(chunks)
    # Coarse pass
    window_scores = {}
    for start in range(0, n, coarse_win):
        win = chunks[start:start + coarse_win]
        summaries = [" ".join(c.split()[:12]) for c in win]
        window_scores[start] = max(bow_score(s, query) for s in summaries)
    
    top_starts = sorted(window_scores, key=window_scores.get, reverse=True)[:fine_k]
    
    # Fine pass
    best_score, best_idx = -1.0, 0
    for start in top_starts:
        win = chunks[start:start + coarse_win]
        w_len = len(win)
        for j, c in enumerate(win):
            s = bow_score(c, query) * attn_weight(j, w_len, strength=0.25) + rng.gauss(0, 0.015)
            if s > best_score:
                best_score, best_idx = s, start + j
    hit = int(best_idx == ev_pos)
    f1 = bow_score(chunks[best_idx], query) * hit
    return float(hit), f1

### 5. Execute Main Simulation ($N=4,000$ Evaluated Retrievals)
Running 200 trials for each of the 4 methods across 5 document locations.

In [ ]:
methods = {
    "Full-Context (Baseline 1)": full_context,
    "Chunked Sequential (Baseline 2)": chunked_sequential,
    "Overlapping Chunks (Baseline 3)": overlapping_chunked,
    "Hierarchical 2-Pass (Proposed)": hierarchical,
}

results = []
print(f"Executing {len(POSITIONS) * len(methods) * NUM_TRIALS} runs across 5 positions...")

for pos_i, (pos, label) in enumerate(zip(POSITIONS, POSITION_LABELS)):
    for mname, mfn in methods.items():
        hits, f1s = [], []
        for t in range(NUM_TRIALS):
            rng = random.Random(RANDOM_SEED + pos_i * 10000 + t)
            query, evidence = build_query_evidence(rng)
            chunks = build_document(evidence, pos, rng)
            h, f = mfn(chunks, query, pos, rng)
            hits.append(h)
            f1s.append(f)
        
        recall = sum(hits) / NUM_TRIALS
        mean_f1 = sum(f1s) / NUM_TRIALS
        results.append({
            "Position": label,
            "Evidence_Index": pos,
            "Method": mname,
            "Recall@1": round(recall, 4),
            "Mean_F1": round(mean_f1, 4),
            "Trials": NUM_TRIALS
        })

df_results = pd.DataFrame(results)
df_pivot = df_results.pivot(index="Position", columns="Method", values="Recall@1")
df_pivot = df_pivot.reindex(POSITION_LABELS)

# Display summary table
print("\n--- Main Results (Recall@1 Table) ---")
display(df_pivot)

### 6. Ablation Experiment: Attention-Decay Sensitivity ($\lambda$)
We vary $\lambda \in [0.0, 3.0]$ at the middle position (50%) to observe degradation onset and plateau.

In [ ]:
pos_mid = POSITIONS[2]
decay_strengths = [0.0, 0.5, 1.0, 1.5, 1.8, 2.0, 2.5, 3.0]
ablation_records = []

for strength in decay_strengths:
    hits = []
    for t in range(NUM_TRIALS):
        rng = random.Random(RANDOM_SEED + 99999 + t)
        query, evidence = build_query_evidence(rng)
        chunks = build_document(evidence, pos_mid, rng)
        n = len(chunks)
        scores = [
            bow_score(c, query) * attn_weight(i, n, strength=strength) + rng.gauss(0, 0.015)
            for i, c in enumerate(chunks)
        ]
        pred = max(range(n), key=lambda i: scores[i])
        hits.append(int(pred == pos_mid))
    rec = sum(hits) / NUM_TRIALS
    ablation_records.append({"Decay_Strength": strength, "Recall@1": round(rec, 4)})

df_ablation = pd.DataFrame(ablation_records)
print("\n--- Ablation Results (Middle Position Recall@1 vs Lambda) ---")
display(df_ablation)

### 7. Publication Figures Generation
These figures match Figures 1, 2, and 3 in the LaTeX ACM Sigconf paper.

In [ ]:
# Figure 1: Recall@1 vs Evidence Position
plt.figure(figsize=(8, 5))
short_names = {
    "Full-Context (Baseline 1)": "FC-B1 (Full Context)",
    "Chunked Sequential (Baseline 2)": "CS-B2 (Sequential Chunking)",
    "Overlapping Chunks (Baseline 3)": "OC-B3 (Overlapping Chunks)",
    "Hierarchical 2-Pass (Proposed)": "H2P (Hierarchical 2-Pass)"
}
markers = {'Full-Context (Baseline 1)': 's', 'Chunked Sequential (Baseline 2)': '^',
           'Overlapping Chunks (Baseline 3)': 'D', 'Hierarchical 2-Pass (Proposed)': 'o'}
colors = {'Full-Context (Baseline 1)': '#d62728', 'Chunked Sequential (Baseline 2)': '#1f77b4',
          'Overlapping Chunks (Baseline 3)': '#2ca02c', 'Hierarchical 2-Pass (Proposed)': '#9467bd'}

for method in methods.keys():
    sub_df = df_results[df_results["Method"] == method]
    plt.plot(sub_df["Position"], sub_df["Recall@1"], marker=markers[method],
             linewidth=2.0, markersize=8, label=short_names[method], color=colors[method])

plt.title("Figure 1: Recall@1 vs. Evidence Position in Long Document", pad=12)
plt.xlabel("Position of Evidence Chunk")
plt.ylabel("Recall@1 Score")
plt.ylim(0.80, 1.02)
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.32), ncol=2, frameon=False)
plt.tight_layout()
plt.savefig("figure1_recall_curve.png", dpi=300)
plt.show()

In [ ]:
# Figure 2: Ablation Bar Chart
plt.figure(figsize=(7, 4.5))
bars = plt.bar([str(s) for s in df_ablation["Decay_Strength"]], df_ablation["Recall@1"],
               color='#4a7ebb', edgecolor='#1b4f8f', width=0.55)
plt.title("Figure 2: Full-Context Recall@1 at Middle Position vs. Attention Decay ($\lambda$)", pad=12)
plt.xlabel(r"Decay Strength $(\lambda)$")
plt.ylabel("Recall@1 (Middle Position)")
plt.ylim(0.85, 1.02)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.005, f"{yval:.2f}", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig("figure2_ablation.png", dpi=300)
plt.show()

In [ ]:
# Figure 3: Average Recall@1 comparison across all positions
avg_recalls = df_results.groupby("Method")["Recall@1"].mean().reindex([
    "Full-Context (Baseline 1)",
    "Chunked Sequential (Baseline 2)",
    "Overlapping Chunks (Baseline 3)",
    "Hierarchical 2-Pass (Proposed)"
])

plt.figure(figsize=(7, 4.5))
bar_colors = ['#d62728', '#1f77b4', '#2ca02c', '#9467bd']
x_labels = ["FC-B1", "CS-B2", "OC-B3", "H2P"]
bars = plt.bar(x_labels, avg_recalls, color=bar_colors, edgecolor='black', width=0.5)
plt.title("Figure 3: Average Recall@1 Across All Document Positions", pad=12)
plt.xlabel("Retrieval Method")
plt.ylabel("Average Recall@1")
plt.ylim(0.92, 1.01)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.003, f"{yval:.4f}", ha='center', va='bottom', fontsize=10, weight='bold')
plt.tight_layout()
plt.savefig("figure3_avg_recall.png", dpi=300)
plt.show()

### 8. Statistical Reliability & Confidence Intervals
Calculation of 95% Wald Confidence Intervals for the binomial proportions ($N=200$):
$$\text{CI}_{95} = \hat{p} \pm 1.96 \sqrt{\frac{\hat{p}(1 - \hat{p})}{N}}$$

In [ ]:
ci_records = []
for idx, row in df_results.iterrows():
    p = row["Recall@1"]
    n = row["Trials"]
    se = math.sqrt((p * (1.0 - p)) / n) if p < 1.0 else 0.0
    margin = 1.96 * se
    ci_records.append({
        "Position": row["Position"],
        "Method": row["Method"],
        "Recall@1": p,
        "95% CI Lower": round(max(0.0, p - margin), 4),
        "95% CI Upper": round(min(1.0, p + margin), 4)
    })

df_ci = pd.DataFrame(ci_records)
# Display records with non-degenerate bounds (FC-B1)
display(df_ci[df_ci["Method"] == "Full-Context (Baseline 1)"])

### 9. Export Artifacts for Reproduction
Export results in CSV format matching the paper's repository.

In [ ]:
df_results.to_csv("results_colab_reproduced.csv", index=False)
df_ablation.to_csv("ablation_colab_reproduced.csv", index=False)
print("Saved results_colab_reproduced.csv and ablation_colab_reproduced.csv successfully.")